In [1]:
import pandas as pd

train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

In [2]:
data_dictionary = {
    "row_id": "identifier",
    "credit_limit": "predictor",
    "sex": "predictor / audit attribute",
    "education": "predictor",
    "marital_status": "predictor",
    "age": "predictor",
    "latest_repayment_status": "predictor",
    "maximum_repayment_delay_6m": "predictor",
    "delayed_months_6m": "predictor",
    "average_bill_6m": "predictor",
    "bill_change_sep_to_apr": "predictor",
    "total_payment_6m": "predictor",
    "payment_to_bill_ratio_6m": "predictor",
    "zero_payment_months_6m": "predictor",
    "utilization_sep": "predictor",
    "repayment_assistance_plan": "predictor",
    "payment_difficulty_next_month": "target",
    "predicted_class": "submission"
}

categorical_columns = [
    "sex",
    "education",
    "marital_status",
    "repayment_assistance_plan",
]

In [3]:
from sklearn.model_selection import train_test_split

predictor_columns = [
    column for column, role in data_dictionary.items() if "predictor" in role
]
target_column = [
    column for column, role in data_dictionary.items() if role == "target"
][0]

# Split only the labeled data. Keep the unlabeled test data untouched for final predictions.
X = train_data[predictor_columns]
y = train_data[target_column]

X_train, X_validation, y_train, y_validation = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=411,
    stratify=y,
)

X_test = test_data[predictor_columns]

In [4]:
# Transform categorical variables into one-hot encoded features

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_cat = encoder.fit_transform(X_train[categorical_columns])
X_validation_cat = encoder.transform(X_validation[categorical_columns])
X_test_cat = encoder.transform(X_test[categorical_columns])

numerical_columns = [
    col for col in predictor_columns
    if col not in categorical_columns
]

X_train_num = X_train[numerical_columns]
X_validation_num = X_validation[numerical_columns]
X_test_num = X_test[numerical_columns]

In [5]:
# Combine the one-hot encoded categorical features and the numerical features into a single feature matrix

encoded_categorical_columns = encoder.get_feature_names_out(categorical_columns)

X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=encoded_categorical_columns,
    index=X_train.index
)

X_validation_cat = pd.DataFrame(
    X_validation_cat,
    columns=encoded_categorical_columns,
    index=X_validation.index
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=encoded_categorical_columns,
    index=X_test.index
)

X_train_encoded = pd.concat(
    [X_train_num, X_train_cat],
    axis=1
)

X_validation_encoded = pd.concat(
    [X_validation_num, X_validation_cat],
    axis=1
)

X_test_encoded = pd.concat(
    [X_test_num, X_test_cat],
    axis=1
)

X_train_encoded.head()

,credit_limit,age,latest_repayment_status,maximum_repayment_delay_6m,delayed_months_6m,average_bill_6m,bill_change_sep_to_apr,total_payment_6m,payment_to_bill_ratio_6m,zero_payment_months_6m,...,education_graduate_school,education_high_school,education_other,education_university,education_unknown,marital_status_married,marital_status_other_or_unknown,marital_status_single,repayment_assistance_plan_no,repayment_assistance_plan_yes
21143,60000,34,2,2,1,59000,-800,13500,0.04,0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
9555,80000,65,0,0,0,51600,77100,9600,0.03,0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
16143,160000,26,-1,0,0,11300,-8100,72400,1.07,0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
22939,20000,23,0,2,1,16400,-500,8000,0.08,1,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
11685,340000,62,0,0,0,495100,37800,116400,0.04,0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [6]:
from interpret import show
from interpret.data import ClassHistogram

hist = ClassHistogram().explain_data(X_train, y_train, name='Train Data')
show(hist)

<!-- http://127.0.0.1:7001/2598952515440/ -->

In [7]:
from interpret.glassbox import ClassificationTree
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

tree = ClassificationTree()
tree.fit(X_train_encoded, y_train)

validation_predictions = tree.predict(X_validation_encoded)
print(f"Validation accuracy: {accuracy_score(y_validation, validation_predictions):.3f}")
print(
    f"Validation balanced accuracy: "
    f"{balanced_accuracy_score(y_validation, validation_predictions):.3f}"
)
print(classification_report(y_validation, validation_predictions))

Validation accuracy: 0.805
Validation balanced accuracy: 0.560
              precision    recall  f1-score   support

           0       0.81      0.98      0.89      5909
           1       0.70      0.14      0.23      1591

    accuracy                           0.80      7500
   macro avg       0.76      0.56      0.56      7500
weighted avg       0.79      0.80      0.75      7500



In [8]:
from interpret import show
tree_global = tree.explain_global(name='Tree')
show(tree_global)

<!-- http://127.0.0.1:7001/2599003075456/ -->

In [9]:
# Classify the unlabeled test data and create a prediction file

test_predictions = tree.predict(X_test_encoded)

submission = test_data[["row_id"]].copy()
submission["predicted_class"] = test_predictions
submission.to_csv("test_predictions.csv", index=False)

submission.head()

,row_id,predicted_class
0,CR26-128173,0
1,CR26-102911,0
2,CR26-104960,0
3,CR26-123913,0
4,CR26-111750,0
